# Phase 3: Constraint-Aware Multi-Objective Bayesian Optimization (Constrained MOBO)
This notebook optimizes the 200 MeV electron injector linac across 3 objectives (\(arepsilon_{nx}, arepsilon_{ny}, \sigma_E\))
while explicitly modeling beam quality constraints (\(\sigma_x, \sigma_y, \sigma_{xp}, \sigma_{yp}, \sigma_z \le 1.0	ext{ mm/mrad}\) and \(195	ext{ MeV} \le E_{	ext{kin}} \le 205	ext{ MeV}\))
using 9 Gaussian Process surrogates and BoTorch constrained \(q	ext{LogNEHVI}\) acquisition.

In [ ]:
# Environment Setup & Imports
%load_ext autoreload
%autoreload 2
import os, sys, time
from concurrent.futures import ThreadPoolExecutor
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

# Set PyTorch default dtype
torch.set_default_dtype(torch.double)

# Ensure project root is in sys.path
sys.path.append("..")
from run_astra import run_astra_simulation
from mobo_utils import evaluate_constrained_objective, compute_ref_point
from file_io import create_run_directory, save_results, save_checkpoint, load_checkpoint
from plot_utils import plot_hypervolume, plot_pareto_objective_space, plot_all_constraints, plot_objective_evolution

from botorch.models import SingleTaskGP, ModelListGP
from botorch.models.transforms.input import Normalize
from botorch.models.transforms.outcome import Standardize
from botorch.fit import fit_gpytorch_mll
from gpytorch.mlls import ExactMarginalLogLikelihood
from botorch.optim import optimize_acqf
from botorch.acquisition.multi_objective.logei import qLogNoisyExpectedHypervolumeImprovement
from botorch.acquisition.multi_objective.objective import IdentityMCMultiOutputObjective
from botorch.utils.multi_objective.box_decompositions.non_dominated import FastNondominatedPartitioning
from botorch.utils.multi_objective.hypervolume import Hypervolume
from botorch.utils.multi_objective.pareto import is_non_dominated


In [ ]:
# Configuration & Parameter Bounds
from astra import Astra
A = Astra("../astra.in")
A.timeout = None
A.verbose = False
A.run()

init_parameters = [
    A["solenoid:maxb(1)"],
    A["quadrupole:q_grad(1)"],
    A["quadrupole:q_grad(2)"],
    A["cavity:phi(1)"],
    A["cavity:phi(2)"],
    A["cavity:phi(4)"],
]

ratio_list = [0.50, 0.50, 0.50, 0.10, 0.10, 0.10]
param_bounds_list = []
for val, r in zip(init_parameters, ratio_list):
    lower_val = val * (1 - r)
    upper_val = val * (1 + r)
    param_bounds_list.append([min(lower_val, upper_val), max(lower_val, upper_val)])

bounds = torch.tensor(param_bounds_list, dtype=torch.double).T
input_transform = Normalize(d=bounds.shape[1], bounds=bounds)
print("Parameter Bounds:
", bounds)


In [ ]:
# Define Constraint Functions on GP outcomes (Y shape ... x 9)
def c_sigma_x(Y): return Y[..., 3] - 1.0e-3
def c_sigma_y(Y): return Y[..., 4] - 1.0e-3
def c_sigma_xp(Y): return Y[..., 5] - 1.0e-3
def c_sigma_yp(Y): return Y[..., 6] - 1.0e-3
def c_sigma_z(Y): return Y[..., 7] - 1.0e-3
def c_E_min(Y): return 195e6 - Y[..., 8]
def c_E_max(Y): return Y[..., 8] - 205e6

CONSTRAINT_FUNCTIONS = [c_sigma_x, c_sigma_y, c_sigma_xp, c_sigma_yp, c_sigma_z, c_E_min, c_E_max]
objective_mapping = IdentityMCMultiOutputObjective(outcomes=[0, 1, 2])


In [ ]:
# Initialize Training Data
num_initial_samples = 16
num_workers = 12
seed = 42
torch.manual_seed(seed)
np.random.seed(seed)

sobol = torch.quasirandom.SobolEngine(dimension=bounds.shape[1], scramble=True, seed=seed)
samples = sobol.draw(num_initial_samples).to(dtype=torch.double)
train_X = bounds[0] + (bounds[1] - bounds[0]) * samples

executor = ThreadPoolExecutor(max_workers=num_workers)
print(f"Evaluating {num_initial_samples} initial samples...")
results = list(executor.map(evaluate_constrained_objective, train_X))

train_Y_list, train_Y_full_list, train_feas_list, initial_constraints_list = zip(*results)
train_Y = torch.stack(train_Y_list)
train_Y_full = torch.stack(train_Y_full_list)
train_feas_mask = torch.stack(train_feas_list)
train_constraints_list = list(initial_constraints_list)

print(f"Initial evaluation done. Feasible samples: {train_feas_mask.sum().item()} / {num_initial_samples}")


In [ ]:
# Constrained MOBO Optimization Loop
n_iterations = 20
q = 8
hypervolumes = []
run_dir = create_run_directory(base_dir="../results")
checkpoint_file = os.path.join(run_dir, "gp_checkpoint", "constrained_mobo_checkpoint.pt")

print(f"Starting Phase 3 Constrained MOBO loop ({n_iterations} iterations, q={q})...")

for iteration in range(n_iterations):
    print(f"
--- Iteration {iteration+1}/{n_iterations} ---")
    # Fit 9 GPs on all data
    gps = [SingleTaskGP(train_X, train_Y_full[:, i:i+1], input_transform=input_transform, outcome_transform=Standardize(m=1)) for i in range(9)]
    model = ModelListGP(*gps)
    for m in model.models:
        mll = ExactMarginalLogLikelihood(m.likelihood, m)
        fit_gpytorch_mll(mll)

    feasible_Y = train_Y[train_feas_mask] if train_feas_mask.sum() > 0 else train_Y
    ref_point = compute_ref_point(feasible_Y)

    acq_func = qLogNoisyExpectedHypervolumeImprovement(
        model=model,
        ref_point=ref_point.tolist(),
        X_baseline=train_X,
        objective=objective_mapping,
        constraints=CONSTRAINT_FUNCTIONS,
        prune_baseline=True
    )

    candidates, _ = optimize_acqf(
        acq_function=acq_func,
        bounds=bounds,
        q=q,
        num_restarts=20,
        raw_samples=128,
        return_best_only=True
    )

    eval_res = list(executor.map(evaluate_constrained_objective, candidates))
    new_Y_list, new_Y_full_list, new_feas_list, new_constraints_tuples = zip(*eval_res)

    train_X = torch.cat([train_X, candidates])
    train_Y = torch.cat([train_Y, torch.stack(new_Y_list)])
    train_Y_full = torch.cat([train_Y_full, torch.stack(new_Y_full_list)])
    train_feas_mask = torch.cat([train_feas_mask, torch.stack(new_feas_list)])
    train_constraints_list.extend(list(new_constraints_tuples))

    feasible_Y_curr = train_Y[train_feas_mask]
    if feasible_Y_curr.shape[0] > 0:
        pareto_mask = is_non_dominated(feasible_Y_curr)
        current_hv = Hypervolume(ref_point=ref_point).compute(feasible_Y_curr[pareto_mask])
    else:
        current_hv = 0.0
    hypervolumes.append(current_hv)
    print(f"  Iteration {iteration+1} done | Feasible in batch: {torch.stack(new_feas_list).sum().item()}/{q} | HV: {current_hv:.4f}")

    save_checkpoint(iteration, train_X, train_Y, train_feas_mask, hypervolumes, train_constraints_list, "qLogNEHVI", checkpoint_file)
    save_results(train_X, train_Y, run_dir, hypervolumes, train_constraints_list)

executor.shutdown()
print("Optimization complete!")


In [ ]:
# Visualization & Analysis
plot_hypervolume(hypervolumes, len(hypervolumes), 0)
plot_pareto_objective_space(train_Y)
plot_all_constraints(train_constraints_list, train_feas_mask)
